In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_oracleworkflow_cepbiplatora(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.Oracle_Workflow"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "cepbiplatora",
            )
        else:
            logging.info("Table does not exist, no need to delete entries for business date!")

        # ================== oracleworkflowtasks (main fact — task level) ==================

        tasks_extra_filters = None

        if tasks_extra_filters is not None:
            tasks_extra_filters.append(date_filter)
            tasks_final_filters = " AND ".join(tasks_extra_filters)
        else:
            tasks_final_filters = date_filter

        tasks_selected_fields = [
            "Number",
            "CreatedBy",
            "FromUserDisplayName",
            "OwnerRole",
            "TaskTitle",
            "Status",
            "State",
            "AssignedDate",
            "CreatedDate",
            "UpdatedById",
            "UpdatedDate",
            "ProcessName",
            "ProcessId",
            "Outcome",
            "Assignee",
            "BusinessDate",
        ]

        tasks = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_oracleworkflowtasks",
            tasks_final_filters,
            tasks_selected_fields,
        )

        # ================== oracleworkflowprocesses (process level) ==================

        processes_extra_filters = None

        if processes_extra_filters is not None:
            processes_extra_filters.append(date_filter)
            processes_final_filters = " AND ".join(processes_extra_filters)
        else:
            processes_final_filters = date_filter

        processes_selected_fields = [
            "ProcessId",
            "ProcessName",
            "OwnedByRole",
            "CreatedBy",
            "State",
            "CreatedDate",
            "UpdatedDate",
            "IsProcessed",
            "BusinessDate",
        ]

        processes = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_oracleworkflowprocesses",
            processes_final_filters,
            processes_selected_fields,
        )

        # ================== rfdtoracleworkflowprocessname (reference lookup) ==================

        processname_extra_filters = None

        if processname_extra_filters is not None:
            processname_extra_filters.append(date_filter)
            processname_final_filters = " AND ".join(processname_extra_filters)
        else:
            processname_final_filters = date_filter

        processname_selected_fields = [
            "ProccessName",
            "Category",
            "StagingDateInserted",
            "BusinessDate",
        ]

        processname_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_rfdtoracleworkflowprocessname",
            processname_final_filters,
            processname_selected_fields,
        )

        # Deduplicate processname reference
        processname_window = Window.partitionBy(
            processname_raw.ProccessName,
            processname_raw.BusinessDate,
        ).orderBy(col("StagingDateInserted").desc())

        processname = (
            processname_raw
            .withColumn("_rn", row_number().over(processname_window))
            .filter(col("_rn") == 1)
            .drop("_rn", "StagingDateInserted")
        )

        # ================== Payload pivot: CDI (PayloadFieldLabelID = 162) ==================

        payload_base_filters = date_filter

        payload_selected_fields = [
            "TaskNumber",
            "PayloadFieldLabelID",
            "Value",
            "BusinessDate",
        ]

        payload_all = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_oracleworkflowtaskspayloadnew",
            payload_base_filters,
            payload_selected_fields,
        )

        # Filter and deduplicate each payload field
        def get_payload_field(payload_df, field_label_id, alias_name):
            """Extract a single payload field, deduplicated per TaskNumber + BusinessDate."""
            filtered = payload_df.filter(col("PayloadFieldLabelID") == field_label_id)
            window = Window.partitionBy("TaskNumber", "BusinessDate").orderBy(col("Value").desc())
            deduped = (
                filtered
                .withColumn("_rn", row_number().over(window))
                .filter(col("_rn") == 1)
                .drop("_rn", "PayloadFieldLabelID")
                .withColumnRenamed("Value", alias_name)
                .withColumnRenamed("TaskNumber", f"{alias_name}_TaskNumber")
                .withColumnRenamed("BusinessDate", f"{alias_name}_BusinessDate")                
            )
            return deduped

        payload_cdi = get_payload_field(payload_all, 162, "CustomerCDI")
        payload_vat = get_payload_field(payload_all, 101, "CustomerAFM")
        payload_contract = get_payload_field(payload_all, 54, "ContractNumber")
        payload_account = get_payload_field(payload_all, 90, "AccountNumber")
        payload_portfolio = get_payload_field(payload_all, 116, "Portfolio")
        payload_spv = get_payload_field(payload_all, 50, "SPVPortfolio")
        payload_source = get_payload_field(payload_all, 104, "ProcessSource")

        # ================== oracleworkflowbpmreport (requests report — process level) ==================

        bpmreport_selected_fields = [
            "ProcessId",
            "Date_of_Reciept",
            "Shipping_Method",
            "Application_Status",
            "BusinessDate",
        ]

        bpmreport_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_oracleworkflowbpmreport",
            date_filter,
            bpmreport_selected_fields,
        )

        # Deduplicate to one row per ProcessId + BusinessDate (report is process-scoped)
        bpmreport_window = Window.partitionBy(
            bpmreport_raw.ProcessId,
            bpmreport_raw.BusinessDate,
        ).orderBy(col("Date_of_Reciept").desc())

        bpmreport = (
            bpmreport_raw
            .withColumn("_rn", row_number().over(bpmreport_window))
            .filter(col("_rn") == 1)
            .drop("_rn")
            .withColumnRenamed("ProcessId", "BpmReport_ProcessId")
            .withColumnRenamed("BusinessDate", "BpmReport_BusinessDate")
            .withColumnRenamed("Date_of_Reciept", "RequestDateOfReceipt")
            .withColumnRenamed("Shipping_Method", "RequestShippingMethod")
            .withColumnRenamed("Application_Status", "RequestApplicationStatus")
        )

        # ================== oracleworkflowcomplaintsreport (complaints report — process level) ==================

        complaintsreport_selected_fields = [
            "Complaint_ID",
            "Date_of_Reciept",
            "Main_Recepient",
            "Sent_by",
            "BusinessDate",
        ]

        complaintsreport_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepbiplatora_oracleworkflowcomplaintsreport",
            date_filter,
            complaintsreport_selected_fields,
        )

        # Deduplicate to one row per Complaint_ID + BusinessDate
        complaintsreport_window = Window.partitionBy(
            complaintsreport_raw.Complaint_ID,
            complaintsreport_raw.BusinessDate,
        ).orderBy(col("Date_of_Reciept").desc())

        complaintsreport = (
            complaintsreport_raw
            .withColumn("_rn", row_number().over(complaintsreport_window))
            .filter(col("_rn") == 1)
            .drop("_rn")
            .withColumnRenamed("Complaint_ID", "ComplaintReport_ProcessId")
            .withColumnRenamed("BusinessDate", "ComplaintReport_BusinessDate")
            .withColumnRenamed("Date_of_Reciept", "ComplaintDateOfReceipt")
            .withColumnRenamed("Main_Recepient", "ComplaintMainRecipient")
            .withColumnRenamed("Sent_by", "ComplaintSentBy")
        )

        # ========================== Joins ==========================

        tasks_processes_join_condition = (
            (tasks.ProcessId == processes.ProcessId)
            & (tasks.BusinessDate == processes.BusinessDate)
        )

        tasks_processname_join_condition = (
            (tasks.ProcessName == processname.ProccessName)
            & (tasks.BusinessDate == processname.BusinessDate)
        )

        tasks_payload_cdi_condition = (
            (tasks.Number == payload_cdi.CustomerCDI_TaskNumber)
            & (tasks.BusinessDate == payload_cdi.CustomerCDI_BusinessDate)
        )

        tasks_payload_vat_condition = (
            (tasks.Number == payload_vat.CustomerAFM_TaskNumber)
            & (tasks.BusinessDate == payload_vat.CustomerAFM_BusinessDate)
        )

        tasks_payload_contract_condition = (
            (tasks.Number == payload_contract.ContractNumber_TaskNumber)
            & (tasks.BusinessDate == payload_contract.ContractNumber_BusinessDate)
        )

        tasks_payload_account_condition = (
            (tasks.Number == payload_account.AccountNumber_TaskNumber)
            & (tasks.BusinessDate == payload_account.AccountNumber_BusinessDate)
        )

        tasks_payload_portfolio_condition = (
            (tasks.Number == payload_portfolio.Portfolio_TaskNumber)
            & (tasks.BusinessDate == payload_portfolio.Portfolio_BusinessDate)
        )

        tasks_payload_spv_condition = (
            (tasks.Number == payload_spv.SPVPortfolio_TaskNumber)
            & (tasks.BusinessDate == payload_spv.SPVPortfolio_BusinessDate)
        )

        tasks_payload_source_condition = (
            (tasks.Number == payload_source.ProcessSource_TaskNumber)
            & (tasks.BusinessDate == payload_source.ProcessSource_BusinessDate)
        )

        tasks_bpmreport_condition = (
            (tasks.ProcessId == bpmreport.BpmReport_ProcessId)
            & (tasks.BusinessDate == bpmreport.BpmReport_BusinessDate)
        )

        tasks_complaintsreport_condition = (
            (tasks.ProcessId == complaintsreport.ComplaintReport_ProcessId)
            & (tasks.BusinessDate == complaintsreport.ComplaintReport_BusinessDate)
        )

        target_table = (
            tasks
            .join(processes, tasks_processes_join_condition, "left")
            .join(processname, tasks_processname_join_condition, "left")
            .join(payload_cdi, tasks_payload_cdi_condition, "left")
            .join(payload_vat, tasks_payload_vat_condition, "left")
            .join(payload_contract, tasks_payload_contract_condition, "left")
            .join(payload_account, tasks_payload_account_condition, "left")
            .join(payload_portfolio, tasks_payload_portfolio_condition, "left")
            .join(payload_spv, tasks_payload_spv_condition, "left")
            .join(payload_source, tasks_payload_source_condition, "left")
            .join(bpmreport, tasks_bpmreport_condition, "left")
            .join(complaintsreport, tasks_complaintsreport_condition, "left")
        )

        # ========================== Primary Key ==========================

        primary_key_fields = [
            tasks.Number,
            tasks.ProcessId,
            lit("cepbiplatora"),
        ]

        target_table = (
            target_table
            .withColumn(
                "OracleWorkflowStepId",
                sha2(concat_ws(";", *primary_key_fields), 256),
            )
            .withColumn(
                "OracleWorkflowStepKey",
                concat_ws(";", tasks.Number, tasks.ProcessId),
            )
        )

        # ========================== Select Final Columns ==========================

        result_df = target_table.select(
            target_table.OracleWorkflowStepId.cast("string").alias("OracleWorkflowStepId"),
            target_table.OracleWorkflowStepKey.cast("string").alias("OracleWorkflowStepKey"),
            lit("cepbiplatora").cast("string").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            tasks.Number.cast("string").alias("TaskNumber"),
            tasks.ProcessId.cast("string").alias("ProcessId"),
            tasks.ProcessName.cast("string").alias("ProcessName"),
            processname.Category.cast("string").alias("ProcessCategory"),
            processes.State.cast("string").alias("ProcessState"),
            processes.OwnedByRole.cast("string").alias("ProcessOwnedByRole"),
            processes.CreatedBy.cast("string").alias("ProcessCreatedBy"),
            processes.CreatedDate.cast("timestamp").alias("ProcessCreatedDate"),
            processes.UpdatedDate.cast("timestamp").alias("ProcessUpdatedDate"),
            processes.IsProcessed.cast("boolean").alias("IsProcessed"),
            tasks.TaskTitle.cast("string").alias("TaskTitle"),
            tasks.Status.cast("string").alias("TaskStatus"),
            tasks.State.cast("string").alias("TaskState"),
            tasks.Outcome.cast("string").alias("TaskOutcome"),
            tasks.CreatedBy.cast("string").alias("TaskCreatedBy"),
            tasks.CreatedDate.cast("timestamp").alias("TaskCreatedDate"),
            tasks.AssignedDate.cast("timestamp").alias("TaskAssignedDate"),
            tasks.UpdatedDate.cast("timestamp").alias("TaskUpdatedDate"),
            tasks.UpdatedById.cast("string").alias("TaskUpdatedBy"),
            tasks.Assignee.cast("string").alias("TaskAssignee"),
            tasks.OwnerRole.cast("string").alias("TaskOwnerRole"),
            tasks.FromUserDisplayName.cast("string").alias("TaskFromUser"),
            payload_cdi.CustomerCDI.cast("string").alias("CustomerCDI"),
            payload_vat.CustomerAFM.cast("string").alias("CustomerAFM"),
            payload_contract.ContractNumber.cast("string").alias("ContractNumber"),
            payload_account.AccountNumber.cast("string").alias("AccountNumber"),
            payload_portfolio.Portfolio.cast("string").alias("Portfolio"),
            payload_spv.SPVPortfolio.cast("string").alias("SPVPortfolio"),
            payload_source.ProcessSource.cast("string").alias("ProcessSource"),
            bpmreport.RequestDateOfReceipt.cast("timestamp").alias("RequestDateOfReceipt"),
            bpmreport.RequestShippingMethod.cast("string").alias("RequestShippingMethod"),
            bpmreport.RequestApplicationStatus.cast("string").alias("RequestApplicationStatus"),
            complaintsreport.ComplaintDateOfReceipt.cast("timestamp").alias("ComplaintDateOfReceipt"),
            complaintsreport.ComplaintMainRecipient.cast("string").alias("ComplaintMainRecipient"),
            complaintsreport.ComplaintSentBy.cast("string").alias("ComplaintSentBy"),
            tasks.BusinessDate.alias("BusinessDate"),
        )

        # ========================== Append =========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_oracleworkflow_cepbiplatora(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)